# Python Variables — From Basics to Top 1% Understanding

Most tutorials stop at *"a variable is a name for a value."* That's true, but it's the surface.

This notebook covers:
1. The basics (assignment, naming rules, dynamic typing)
2. **What a variable actually is under the hood** (names bound to objects, not boxes holding values)
3. **Object identity vs equality** (`is` vs `==`), and why this trips people up
4. **CPython implementation details**: small-int caching, string interning
5. **Mutability vs reassignment** — the #1 source of subtle bugs
6. Reference counting & garbage collection
7. Scope (`global`, `nonlocal`) and the classic mutable-default-argument trap
8. Modern syntax: the walrus operator, type hints
9. A quiz to test your understanding

Run every cell yourself — this topic is best understood by watching `id()` and `is` in action, not by reading about them.

## 1. The Basics

In [1]:
x = 5
name = "Alex"
print(x)
print(name)
print(type(x), type(name))

5
Alex
<class 'int'> <class 'str'>


Python is **dynamically typed**: you never declare a type. The type lives on the *object*, not on the variable name.

In [2]:
x = 10
print(type(x))
x = "Now a string"
print(type(x))

<class 'int'>
<class 'str'>


### Naming rules

- Letters, digits, underscores; can't start with a digit.
- Case-sensitive (`myVar` != `myvar`).
- Can't use reserved keywords (`if`, `else`, `for`, `class`, ...).

Valid: `age`, `_colour`, `total_score`

Invalid: `1name`, `class`, `user-name`

**Top 1% tip:** you can check *all* reserved keywords programmatically instead of memorizing a list.

In [3]:
import keyword
print(keyword.kwlist)
print(keyword.iskeyword("class"))
print(keyword.iskeyword("total_score"))

['False', 'None', 'True', 'and', 'as', 'assert', 'async', 'await', 'break', 'class', 'continue', 'def', 'del', 'elif', 'else', 'except', 'finally', 'for', 'from', 'global', 'if', 'import', 'in', 'is', 'lambda', 'nonlocal', 'not', 'or', 'pass', 'raise', 'return', 'try', 'while', 'with', 'yield']
True
False


### Multiple assignment

In [2]:
# Same value to multiple names
a = b = c = 100
print(a, b, c)

# Different values, one line (tuple unpacking under the hood)
x, y, z = 1, 2.5, "Python"
print(x, y, z)

100 100 100
1 2.5 Python
<class 'str'>


## 2. What a Variable *Actually* Is

This is the idea that separates people who *use* Python from people who *understand* it.

In C or Java, a variable is a labeled box: the box has a fixed type and holds a value directly.

In Python, a variable is just a **name in a namespace (dictionary) that points to an object living somewhere in memory**. Assignment (`=`) never copies data — it just makes a name point at an object. This is why Python people say "binding" instead of "storing."

You can literally see the namespace:

In [5]:
x = 5
name = "Alex"

# globals() is the actual dict Python uses to look up module-level names
print(globals()["x"])
print(globals()["name"])

5
Alex


### `id()` and `is`: seeing object identity directly

`id(obj)` returns a unique integer identifying the object (in CPython, it's the memory address). `is` checks whether two names point to the **same object**, not whether their values are equal. This is the tool for verifying everything below.

In [6]:
x = 5
print("id(x):", id(x))

y = x  # y now points to the SAME object as x
print("id(y):", id(y))
print("x is y:", x is y)   # True: same object
print("x == y:", x == y)   # True: same value too

x = 'Geeks'  # x is rebound to a NEW object
print("\nAfter x = 'Geeks':")
print("id(x):", id(x))
print("y is still:", y)     # y is untouched, still 5
print("x is y:", x is y)    # False now

id(x): 11755816
id(y): 11755816
x is y: True
x == y: True

After x = 'Geeks':
id(x): 140247441057920
y is still: 5
x is y: False


This matches the article's "shared reference" diagram exactly — but now you can *prove* it with `id()` instead of trusting a picture.

## 3. CPython Implementation Quirks Almost Nobody Knows

These are genuinely "top 1%" facts — they explain behavior that confuses even experienced developers.

### Small integer caching

CPython pre-creates and caches all integers from **-5 to 256** at startup. Any variable assigned one of these values just points to the same cached object — that's why `is` "works" for small ints even though you're not supposed to rely on it.

In [7]:
a = 100
b = 100
print("100 is 100:", a is b)   # True -- cached small int

c = 1000
d = 1000
print("1000 is 1000:", c is d) # Often False -- NOT cached, separate objects
print("But equal:", c == d)    # Always True

100 is 100: True
1000 is 1000: False
But equal: True


**The lesson:** never use `is` to compare values (use `==`). Only use `is` to check identity — most commonly `x is None`.

### String interning

Similarly, CPython "interns" (reuses) certain strings — typically short identifier-like strings — so equal literals may share one object. This is an optimization detail, not a guarantee, and differs by Python build.

In [8]:
s1 = "hello"
s2 = "hello"
print("s1 is s2:", s1 is s2)  # Often True: interned

s3 = "hello world!"
s4 = "hello world!"
print("s3 is s4:", s3 is s4)  # Often False: not auto-interned (spaces/punctuation)

import sys
s5 = sys.intern("hello world!")
s6 = sys.intern("hello world!")
print("Forced interning:", s5 is s6)  # True: you can force it

s1 is s2: True
s3 is s4: False
Forced interning: True


## 4. Mutability vs. Reassignment — the Real Source of Bugs

This is the single most important distinction for writing correct Python.

- **Reassignment** (`x = new_value`) changes what a *name* points to. It never affects other names.
- **Mutation** (`obj.append(...)`, `obj[0] = ...`) changes the *object itself in place*. Every name pointing to that object sees the change.

Immutable types: `int`, `float`, `str`, `tuple`, `bool`, `frozenset`.

Mutable types: `list`, `dict`, `set`, and most custom objects.

The article's example only used ints (immutable), which is why `y = x; y = y + 1` left `x` untouched. Swap in a **mutable** object and the behavior flips completely:

In [9]:
# Immutable case (matches the article) -- reassignment, no shared effect
x = 1
y = x
y = y + 1
print("Immutable -> x:", x, " y:", y)  # x unaffected

print()

# Mutable case -- MUTATION, shared effect!
list_x = [1, 2, 3]
list_y = list_x       # list_y points to the SAME list object
list_y.append(4)       # mutates the object in place
print("Mutable -> list_x:", list_x)  # list_x is ALSO changed!
print("Mutable -> list_y:", list_y)
print("Same object?", list_x is list_y)

Immutable -> x: 1  y: 2

Mutable -> list_x: [1, 2, 3, 4]
Mutable -> list_y: [1, 2, 3, 4]
Same object? True


This single distinction (mutation vs. rebinding) explains a huge fraction of "why did my other variable change too?!" bugs in real codebases. If you only remember one thing from this notebook, remember this cell.

### Fixing it: make a copy when you want independence

In [9]:
list_x = [1, 2, 3]
list_y = list_x.copy()   # or list(list_x), or list_x[:]
list_y.append(4)
print("list_x:", list_x)  # untouched now
print("list_y:", list_y)

# Note: .copy() is SHALLOW. For nested structures use copy.deepcopy
import copy
nested = [[1, 2], [3, 4]]
shallow = nested.copy()
shallow[0].append(99)
print("shallow copy leaks into original nested list:", nested)

deep = copy.deepcopy(nested)
deep[0].append(-1)
print("deepcopy does NOT leak:", nested)

print(list_x)
print(nested)
print()

list_x: [1, 2, 3]
list_y: [1, 2, 3, 4]
shallow copy leaks into original nested list: [[1, 2, 99], [3, 4]]
deepcopy does NOT leak: [[1, 2, 99], [3, 4]]
[1, 2, 3]
[[1, 2, 99], [3, 4]]



## 5. Reference Counting & Garbage Collection

CPython uses **reference counting** as its primary memory management strategy. Every object tracks how many names/containers point to it; when that count hits zero, the object is freed immediately (a cyclic garbage collector handles reference cycles separately).

In [11]:
import sys

a = [1, 2, 3]
print("refcount right after creation:", sys.getrefcount(a) - 1)  # -1: getrefcount itself adds a temp ref

b = a
print("refcount after b = a:", sys.getrefcount(a) - 1)

del b
print("refcount after del b:", sys.getrefcount(a) - 1)

refcount right after creation: 1
refcount after b = a: 2
refcount after del b: 1


In [12]:
# The article's del example
x = 10
del x
try:
    print(x)
except NameError as e:
    print("NameError:", e)

NameError: name 'x' is not defined


## 6. Scope: `global` and `nonlocal`

A variable's scope is determined by *where it's assigned*, following the **LEGB rule**: Local -> Enclosing -> Global -> Built-in.

In [13]:
counter = 0  # global

def increment_broken():
    counter += 1   # UnboundLocalError: assignment makes it local by default

def increment_fixed():
    global counter
    counter += 1

try:
    increment_broken()
except UnboundLocalError as e:
    print("Broken version fails:", e)

increment_fixed()
print("Fixed version, counter is now:", counter)

Broken version fails: cannot access local variable 'counter' where it is not associated with a value
Fixed version, counter is now: 1


In [12]:
def outer():
    total = 0
    def inner():
        nonlocal total   # binds to the ENCLOSING function's variable, not global
        total += 5
    inner()
    return total
print(outer())

5


## 7. The Classic Trap: Mutable Default Arguments

Default argument values are evaluated **once**, when the function is defined — not on every call. If the default is mutable, every call that doesn't supply that argument shares the *same* object.

In [15]:
def append_bad(item, bucket=[]):   # DANGER: shared default list
    bucket.append(item)
    return bucket

print(append_bad(1))
print(append_bad(2))   # you'd expect [2], but the list from the first call is reused!

def append_good(item, bucket=None):
    if bucket is None:
        bucket = []
    bucket.append(item)
    return bucket

print(append_good(1))
print(append_good(2))

[1]
[1, 2]
[1]
[2]


## 8. Modern Syntax

### The walrus operator `:=` (Python 3.8+)
Assigns and returns a value in the same expression — useful for avoiding duplicate work.

In [16]:
data = [1, 2, 3, 4, 5, 6, 7, 8]

# Without walrus: compute len(data) twice, or use an extra line
if len(data) > 5:
    print(f"Big list, size {len(data)}")

# With walrus: compute once, bind, and use
if (n := len(data)) > 5:
    print(f"Big list, size {n}")

Big list, size 8
Big list, size 8


### Type hints
Python stays dynamically typed at runtime, but you can *annotate* variables for readability and static tools like `mypy`. This is what production/professional codebases use.

In [17]:
age: int = 21
name: str = "Alex"
scores: list[int] = [90, 85, 88]

# Hints are NOT enforced at runtime -- this still "works" (but a type checker would flag it)
age = "twenty-one"
print(age, type(age))

print("__annotations__:", __annotations__)

twenty-one <class 'str'>
__annotations__: {'age': <class 'int'>, 'name': <class 'str'>, 'scores': list[int]}


## 9. Practical Patterns from the Article

In [18]:
# Swapping without a temp variable
a, b = 5, 10
a, b = b, a
print(a, b)

# Under the hood: Python builds a tuple (b, a) on the right FIRST,
# then unpacks it into (a, b) on the left -- that's why no temp var is needed.

# Counting characters
word = "Python"
length = len(word)
print("Length of the word:", length)

10 5
Length of the word: 6


## 10. Quick Self-Check

Predict the output *before* running each cell, then check yourself.

In [19]:
# Q1: What prints?
a = [1, 2, 3]
b = a
a = a + [4]   # note: + creates a NEW list, unlike .append()
print(a, b)

[1, 2, 3, 4] [1, 2, 3]


In [15]:
# Q2: What prints?
def f(x=[]):
    x.append(1)
    return x

print(f())
print(f())
print(f([9]))
print(f([9,9,9]))
print(f())

[1]
[1, 1]
[9, 1]
[9, 9, 9, 1]
[1, 1, 1]


In [21]:
# Q3: True or False?
x = 300
y = 300
print(x == y)
print(x is y)

True
False


---
### Answers
**Q1:** `a = [1, 2, 3, 4]`, `b = [1, 2, 3]` — `a + [4]` builds a new list; `a` is rebound, `b` still points to the original.

**Q2:** `[1]`, `[1, 1]`, `[9, 1]`, `[1, 1, 1]` — the default list is created once and shared across calls that don't pass their own list.

**Q3:** `True`, then almost always `False` — 300 is outside the -5..256 cached small-int range, so CPython typically creates two distinct objects with equal *value* but different *identity*.

---
## Summary

| Concept | Beginner takeaway | Top 1% takeaway |
|---|---|---|
| Variable | A name for a value | A name bound in a namespace to an object |
| `=` | Assignment | Rebinding a name — never copies |
| `is` vs `==` | "Same thing" | `is` = identity, `==` = equality; only use `is` for identity checks (e.g. `is None`) |
| Reassignment | Changes the variable | Only changes what *that name* points to |
| Mutation | "Changing a variable" | Changes the *object*, visible through every name pointing to it |
| Default args | Convenience | Evaluated once at def-time — never use mutable defaults |
| GC | "Python cleans up" | Reference counting + cycle detector; `sys.getrefcount` makes it visible |

# Equality Checking Across Python Data Types

In Python, `==` is the primary operator for checking **value equality**, but different data types have their own subtleties.

| Type                | How to Check Equality | Gotcha                                                                                                           |
| ------------------- | --------------------- | ---------------------------------------------------------------------------------------------------------------- |
| `int`               | `a == b`              | Exact value comparison; no special handling is normally needed.                                                  |
| `float`             | `math.isclose(a, b)`  | `==` can fail because of floating-point rounding, e.g. `0.1 + 0.2 != 0.3`.                                       |
| `bool`              | `a == b`              | `True == 1` and `False == 0` are both `True`.                                                                    |
| `str`               | `a == b`              | Case-sensitive. Use `.strip()`, `.lower()`, or `.casefold()` if appropriate.                                     |
| `complex`           | `a == b`              | Compares both the real and imaginary parts.                                                                      |
| `list`              | `a == b`              | Order matters: `[1, 2] != [2, 1]`.                                                                               |
| `tuple`             | `a == b`              | Order matters, just like with lists.                                                                             |
| `dict`              | `a == b`              | Dictionary order does not affect equality; key-value pairs are compared.                                         |
| `set` / `frozenset` | `a == b`              | Order does not matter because sets are unordered collections.                                                    |
| `None`              | `x is None`           | Prefer `is None` for checking against `None`.                                                                    |
| `bytes`             | `a == b`              | Compares the byte sequences for equality.                                                                        |
| `float('nan')`      | `math.isnan(x)`       | `nan == nan` is always `False` according to IEEE floating-point behavior.                                        |
| Custom objects      | `a == b`              | Uses the class's `__eq__()` method if defined. Otherwise, object identity is used by the default implementation. |

## Quick Takeaway

* Use **`==`** for value equality in most cases.
* Use **`math.isclose()`** when comparing floating-point numbers.
* Use **`is None`** when checking whether a value is `None`.
* Use **`math.isnan()`** to check whether a floating-point value is `NaN`.
* For custom classes, `==` depends on how `__eq__()` is implemented.
